# FlashAttention from scratch — Notebook 01

Goal: derive standard attention, implement online softmax, then tiled exact attention. Start with NumPy float64 for easy verification.

For a single query, $o=\sum_j e^{s_j}v_j/\sum_j e^{s_j}$, where $s_j=q^Tk_j/\sqrt d$. We maintain a running maximum $m$, denominator $\ell$, and weighted numerator $a$.

In [8]:
import numpy as np
rng = np.random.default_rng(42)
N, d, dv = 13, 8, 6
Q = rng.normal(size=(N,d))
K = rng.normal(size=(N,d))
V = rng.normal(size=(N,dv))

## 1. Reference implementation
Stable softmax subtracts each row maximum. This implementation explicitly materializes the $N\times N$ score matrix.

In [9]:
def attention_reference(Q, K, V):
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights @ V

expected = attention_reference(Q, K, V)
print(expected.shape, expected[:2])

(13, 6) [[ 0.34796395 -0.11834042  0.11082018 -0.18081038 -0.62338161 -0.22994395]
 [ 0.45120362 -0.06779337 -0.1373039   0.18554955  0.16751714 -0.23758034]]


## 2. Online softmax for one query
For a new score $s$, update $m'=\max(m,s)$, $\ell'=e^{m-m'}\ell+e^{s-m'}$, $a'=e^{m-m'}a+e^{s-m'}v$.

In [10]:
def attention_one_query_online(q, K, V):
    m, ell = -np.inf, 0.0
    a = np.zeros(V.shape[-1], dtype=np.float64)
    for k, v in zip(K, V):
        s = np.dot(q, k) / np.sqrt(q.size)
        new_m = max(m, s)
        alpha = np.exp(m - new_m) if np.isfinite(m) else 0.0
        beta = np.exp(s - new_m)
        ell = alpha * ell + beta
        a = alpha * a + beta * v
        m = new_m
    return a / ell

actual = np.stack([attention_one_query_online(q, K, V) for q in Q])
np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
print("Online attention matches reference")

Online attention matches reference


## 3. Tiled FlashAttention forward (educational CPU version)
Process query blocks and key/value blocks. Keep row-wise running max $m$, normalization $\ell$, and unnormalized output accumulator $a$. No full $N\times N$ matrix is created. This is algorithmic, not yet a fast GPU kernel.

In [11]:
def attention_tiled(Q, K, V, block_q=4, block_k=5):
    nq, d = Q.shape
    nk = K.shape[0]
    out = np.empty((nq, V.shape[1]), dtype=np.float64)
    for i in range(0, nq, block_q):
        qb = Q[i:i+block_q]
        bq = qb.shape[0]
        m = np.full((bq,), -np.inf)
        ell = np.zeros((bq,))
        a = np.zeros((bq, V.shape[1]))
        for j in range(0, nk, block_k):
            kb, vb = K[j:j+block_k], V[j:j+block_k]
            scores = qb @ kb.T / np.sqrt(d)
            tile_m = scores.max(axis=1)
            new_m = np.maximum(m, tile_m)
            alpha = np.exp(np.where(np.isfinite(m), m-new_m, -np.inf))
            p = np.exp(scores - new_m[:, None])
            ell = alpha * ell + p.sum(axis=1)
            a = alpha[:, None] * a + p @ vb
            m = new_m
        out[i:i+bq] = a / ell[:, None]
    return out

for bq in [1, 3, 8]:
    for bk in [1, 4, 7]:
        np.testing.assert_allclose(attention_tiled(Q,K,V,bq,bk), expected, rtol=1e-12, atol=1e-12)
print("All tile configurations match reference")

All tile configurations match reference


## 4. Numerical stability experiment
Adding the same constant to all scores for a query does not change softmax, but naive exponentiation can overflow. Test with large query magnitudes.

In [12]:
Q_big = Q * 1000
ref_big = attention_reference(Q_big, K, V)
tiled_big = attention_tiled(Q_big, K, V)
np.testing.assert_allclose(tiled_big, ref_big, rtol=1e-11, atol=1e-11)
print("Large-score stability test passed")

Large-score stability test passed


## 5. Experiments to try
1. Change sequence length and tile sizes, including nonmultiples of tile sizes.
2. Measure memory: reference scores require $N^2$ elements; tiled scores require at most `block_q * block_k` elements (plus Q/K/V/output).
3. Implement causal masking (ensure fully masked tiles are handled).
4. Port to PyTorch, then a fused Triton/CUDA kernel.
5. Later: backward pass and benchmark against `torch.nn.functional.scaled_dot_product_attention`.

**Note:** Python/NumPy tiling demonstrates correctness, not GPU speedup; actual FlashAttention performance depends on fused kernels, on-chip memory, and GPU-specific tuning.

In [6]:
import platform
import torch

print("Processor:", platform.processor())
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

Q = torch.randn(128, 64, device=device)
K = torch.randn(128, 64, device=device)
V = torch.randn(128, 64, device=device)

scores = Q @ K.T / (64 ** 0.5)
attention = torch.softmax(scores, dim=-1)
output = attention @ V

print("Device:", output.device)
print("Output shape:", output.shape)

Processor: arm
PyTorch: 2.14.1
MPS available: True
Device: mps:0
Output shape: torch.Size([128, 64])
